# Benchmark: Constraint Mode Comparison

Compares the four constraint configurations on the same deformation fields:

| Mode | Flags |
|------|-------|
| **Jacobian only** | default |
| **Jacobian + Shoelace** | `enforce_shoelace=True` |
| **Jacobian + Injectivity** | `enforce_injectivity=True` |
| **All constraints** | both `True` |

Metrics: runtime, L2 error, final min Jdet, SLSQP iterations (outer), and
whether all negative Jacobians were eliminated.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

from dvfopt import (
    iterative_with_jacobians2,
    jacobian_det2D,
    generate_random_dvf,
    scale_dvf,
)
from dvfopt.testcases import SYNTHETIC_CASES, make_deformation, make_random_dvf
from dvfopt.viz import plot_deformations

## Constraint modes

In [ ]:
MODES = {
    "Jacobian only": {"enforce_shoelace": False, "enforce_injectivity": False},
    "Jac + Shoelace": {"enforce_shoelace": True, "enforce_injectivity": False},
    "Jac + Injectivity": {"enforce_shoelace": False, "enforce_injectivity": True},
    "All constraints": {"enforce_shoelace": True, "enforce_injectivity": True},
}

## Helper: run one test case across all modes

In [ ]:
def run_case(deformation_i, label, n_runs=1):
    """Run all constraint modes on a single deformation field."""
    phi_init = np.stack([deformation_i[-2, 0], deformation_i[-1, 0]])
    jac_init = jacobian_det2D(phi_init)
    n_neg_init = int((jac_init <= 0).sum())
    H, W = deformation_i.shape[-2:]

    print(f"\n{'='*80}")
    print(f"  {label}  |  {H}x{W}  |  Initial neg-Jdet: {n_neg_init}")
    print(f"{'='*80}")

    results = {}
    for mode_name, flags in MODES.items():
        times = []
        for _ in range(n_runs):
            t0 = time.perf_counter()
            phi = iterative_with_jacobians2(
                deformation_i.copy(), verbose=0, **flags
            )
            times.append(time.perf_counter() - t0)

        jac_final = jacobian_det2D(phi)
        final_neg = int((jac_final <= 0).sum())
        final_min = float(jac_final.min())
        l2_err = float(np.sqrt(np.sum((phi - phi_init) ** 2)))
        avg_t = np.mean(times)

        results[mode_name] = {
            "time": avg_t,
            "final_neg": final_neg,
            "final_min": final_min,
            "l2_err": l2_err,
            "phi": phi,
        }

        status = "OK" if final_neg == 0 else "WARN"
        print(f"  {mode_name:<20s}  {avg_t:8.2f}s  "
              f"neg={final_neg:3d}  min_jdet={final_min:+.6f}  "
              f"L2={l2_err:.4f}  [{status}]")

    return results

---
## Test Cases

In [ ]:
all_results = {}

# --- Synthetic correspondence cases ---
for key in ["01a_10x10_crossing", "01c_20x40_edges", "03d_20x20_crossing"]:
    deformation, ms, fs = make_deformation(key)
    title = SYNTHETIC_CASES[key]["title"]
    all_results[title] = run_case(deformation, title)

In [ ]:
# --- Random DVF cases ---
for key in ["01e_20x20_random_spirals", "01f_20x20_random_seed_42"]:
    deformation = make_random_dvf(key)
    from dvfopt.testcases import RANDOM_DVF_CASES
    title = RANDOM_DVF_CASES[key]["title"]
    all_results[title] = run_case(deformation, title)

In [ ]:
# --- Real-data slice (if available) ---
import os
real_path = "../data/test_cases/02b_64x91_slice200.npy"
if os.path.exists(real_path):
    deformation = np.load(real_path)
    all_results["Real 64x91"] = run_case(deformation, "Real 64x91 slice 200")
else:
    print(f"Skipping real data — {real_path} not found")

---
## Summary Table

In [ ]:
mode_names = list(MODES.keys())

print(f"{'Test Case':<28s}", end="")
for m in mode_names:
    print(f"  {m:>20s}", end="")
print()
print("-" * (28 + 22 * len(mode_names)))

for label, results in all_results.items():
    # Time row
    print(f"{label:<28s}", end="")
    for m in mode_names:
        r = results[m]
        print(f"  {r['time']:>17.2f}s  ", end="")
    print("  (time)")

    # L2 row
    print(f"{'':28s}", end="")
    for m in mode_names:
        r = results[m]
        print(f"  {r['l2_err']:>18.4f} ", end="")
    print("  (L2)")

    # Min Jdet row
    print(f"{'':28s}", end="")
    for m in mode_names:
        r = results[m]
        print(f"  {r['final_min']:>+18.6f}", end="")
    print("  (min Jdet)")
    print()

## Plots

In [ ]:
# --- Bar chart: runtime by mode ---
case_labels = list(all_results.keys())
x = np.arange(len(case_labels))
width = 0.18

fig, ax = plt.subplots(figsize=(12, 5))
for i, m in enumerate(mode_names):
    times = [all_results[c][m]["time"] for c in case_labels]
    ax.bar(x + i * width, times, width, label=m)

ax.set_ylabel("Time (s)")
ax.set_title("Runtime by Constraint Mode")
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(case_labels, rotation=25, ha="right")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Bar chart: L2 error by mode ---
fig, ax = plt.subplots(figsize=(12, 5))
for i, m in enumerate(mode_names):
    l2s = [all_results[c][m]["l2_err"] for c in case_labels]
    ax.bar(x + i * width, l2s, width, label=m)

ax.set_ylabel("L2 Error")
ax.set_title("L2 Deviation by Constraint Mode")
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(case_labels, rotation=25, ha="right")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Scatter: runtime vs L2 (Pareto front) ---
fig, ax = plt.subplots(figsize=(8, 6))
markers = ["o", "s", "^", "D"]

for i, m in enumerate(mode_names):
    times = [all_results[c][m]["time"] for c in case_labels]
    l2s = [all_results[c][m]["l2_err"] for c in case_labels]
    ax.scatter(times, l2s, marker=markers[i], s=80, label=m, zorder=3)

ax.set_xlabel("Time (s)")
ax.set_ylabel("L2 Error")
ax.set_title("Runtime vs L2 Error — Constraint Modes")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()